# Chunking and Embedding — the Basics

One book, three chunking strategies, and a look at what actually comes back
when you search. No benchmark, no ground-truth labels, no metric.

**You already know this story**, which is the point: when you ask *"what
happens at the tea party?"* you can tell instantly whether the retrieved text
is right, just by reading it.

The whole book is 36,926 tokens, so embedding it costs about **$0.0007**.
Re-run any cell as often as you like — sweeping ten chunk sizes costs under a
cent.

When you want *measured* answers instead of eyeballed ones, go to
[science_qa.ipynb](science_qa.ipynb). Build the intuition here first.

## 1. Get the book

The entire data-loading step. No dataset library, no schema.

In [1]:
import re

import numpy as np
import requests
import tiktoken
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)
ai = OpenAI()

ENC = tiktoken.get_encoding("cl100k_base")
ntok = lambda t: len(ENC.encode(t))

In [3]:
url = "https://www.gutenberg.org/files/11/11-0.txt"   # Alice in Wonderland
resp = requests.get(url, timeout=30)
resp.encoding = "utf-8"
raw = resp.text

# Strip Project Gutenberg's legal header and footer
start = raw.index("*** START")
end = raw.index("*** END")
book = raw[raw.index("\n", start) + 1 : end].strip()

print(f"{len(book):,} characters   {ntok(book):,} tokens")
print(f"{len(re.findall(r'^CHAPTER [IVX]+', book, re.M))} chapters")
print()
print(book[:1000])

144,599 characters   36,926 tokens
12 chapters

[Illustration]




Alice’s Adventures in Wonderland

by Lewis Carroll

THE MILLENNIUM FULCRUM EDITION 3.0

Contents

 CHAPTER I.     Down the Rabbit-Hole
 CHAPTER II.    The Pool of Tears
 CHAPTER III.   A Caucus-Race and a Long Tale
 CHAPTER IV.    The Rabbit Sends in a Little Bill
 CHAPTER V.     Advice from a Caterpillar
 CHAPTER VI.    Pig and Pepper
 CHAPTER VII.   A Mad Tea-Party
 CHAPTER VIII.  The Queen’s Croquet-Ground
 CHAPTER IX.    The Mock Turtle’s Story
 CHAPTER X.     The Lobster Quadrille
 CHAPTER XI.    Who Stole the Tarts?
 CHAPTER XII.   Alice’s Evidence




CHAPTER I.
Down the Rabbit-Hole


Alice was beginning to get very tired of sitting by her sister on the
bank, and of having nothing to do: once or twice she had peeped into
the book her sister was reading, but it had no pictures or
conversations in it, “and what is the use of a book,” thought Alice
“without pictures or conversations?”

So she was considering in her 

## 2. Why chunk at all?

Two reasons, both concrete:

1. **The embedding model has an input limit** — 8,192 tokens. This book is
   36,926, so it physically cannot be embedded in one piece.
2. **Even if it fit, one vector for a whole book is useless.** An embedding
   is an average of everything in the text. A single vector for all of Alice
   would be "about" wonderland in general, and equally close to every
   question you could ask. It could never point you at the tea party
   specifically.

Chunking splits the book into pieces small enough that each vector is *about
one thing*.

In [4]:
# What does a whole-book embedding even look like? Try it and see it refused.
try:
    ai.embeddings.create(model="text-embedding-3-small", input=[book])
except Exception as e:
    print(type(e).__name__)
    print(str(e)[:200])

BadRequestError
Error code: 400 - {'error': {'message': "Invalid 'input[0]': maximum input length is 8192 tokens.", 'type': 'invalid_request_error', 'param': None, 'code': None}}


## 3. Three ways to cut it up

Every chunker below takes text and returns a list of strings. That is all a
chunker is.

In [4]:
def chunk_fixed(text, size=200, overlap=0):
    """Cut every `size` tokens, regardless of where sentences fall."""
    ids = ENC.encode(text)
    step = size - overlap
    out = []
    for i in range(0, len(ids), step):
        piece = ids[i : i + size]
        if not piece:
            break
        out.append(ENC.decode(piece))
        if i + size >= len(ids):
            break
    return out


def chunk_paragraphs(text, target=200):
    """Group whole paragraphs together until they reach roughly `target` tokens."""
    paras = [p.strip() for p in re.split(r"\n\s*\n", text) if p.strip()]
    out, buf, n = [], [], 0
    for p in paras:
        t = ntok(p)
        if buf and n + t > target:
            out.append("\n\n".join(buf))
            buf, n = [], 0
        buf.append(p)
        n += t
    if buf:
        out.append("\n\n".join(buf))
    return out


def chunk_chapters(text):
    """One chunk per chapter -- natural, but far too big to be useful."""
    parts = re.split(r"(?=^CHAPTER [IVX]+)", text, flags=re.M)
    return [p.strip() for p in parts if p.strip()]

In [5]:
for name, chunks in [
    ("fixed-200",       chunk_fixed(book, 200)),
    ("fixed-200/ov50",  chunk_fixed(book, 200, 50)),
    ("paragraphs-200",  chunk_paragraphs(book, 200)),
    ("chapters",        chunk_chapters(book)),
]:
    sizes = [ntok(c) for c in chunks]
    print(f"{name:18s} {len(chunks):>4} chunks   "
          f"median {int(np.median(sizes)):>4} tok   max {max(sizes):>5} tok")

fixed-200           185 chunks   median  200 tok   max   200 tok
fixed-200/ov50      246 chunks   median  200 tok   max   200 tok
paragraphs-200      224 chunks   median  175 tok   max   272 tok
chapters             13 chunks   median 2949 tok   max  3581 tok


## 4. Look at the seams

This is the part worth staring at. Fixed-size chunking cuts wherever the
token counter runs out — often mid-sentence, sometimes mid-word.

In [6]:
fixed = chunk_fixed(book, 200)

print("=" * 72)
print("END of chunk 5:")
print("=" * 72)
print("..." + fixed[5][-220:])
print()
print("=" * 72)
print("START of chunk 6  <- the sentence above continues here")
print("=" * 72)
print(fixed[6][:220] + "...")

END of chunk 5:
...ll think me for asking! No, it’ll never do
to ask: perhaps I shall see it written up somewhere.”

Down, down, down. There was nothing else to do, so Alice soon began
talking again. “Dinah’ll miss me very much to-night, I

START of chunk 6  <- the sentence above continues here
 should think!”
(Dinah was the cat.) “I hope they’ll remember her saucer of milk at
tea-time. Dinah my dear! I wish you were down here with me! There are
no mice in the air, I’m afraid, but you might catch a bat, and tha...


In [7]:
# Paragraph chunking never splits a sentence, because it only cuts at blank lines.
paras = chunk_paragraphs(book, 200)

print("=" * 72)
print("END of paragraph-chunk 5:")
print("=" * 72)
print("..." + paras[5][-220:])
print()
print("=" * 72)
print("START of paragraph-chunk 6  <- clean break")
print("=" * 72)
print(paras[6][:220] + "...")

END of paragraph-chunk 5:
...r such a fall as this, I shall
think nothing of tumbling down stairs! How brave they’ll all think me
at home! Why, I wouldn’t say anything about it, even if I fell off the
top of the house!” (Which was very likely true.)

START of paragraph-chunk 6  <- clean break
Down, down, down. Would the fall _never_ come to an end? “I wonder how
many miles I’ve fallen by this time?” she said aloud. “I must be
getting somewhere near the centre of the earth. Let me see: that would
be four thous...


### What overlap does

Overlap repeats the tail of each chunk at the head of the next, so a sentence
cut in half still appears whole *somewhere*. The cost is duplication: more
chunks, more vectors, more money.

In [8]:
plain = chunk_fixed(book, 200, 0)
over = chunk_fixed(book, 200, 50)
print(f"no overlap : {len(plain)} chunks")
print(f"50 overlap : {len(over)} chunks  (+{100 * (len(over) / len(plain) - 1):.0f}% to store and embed)")
print()
print("The last 50 tokens of chunk 3 reappear at the start of chunk 4:")
print("---")
print(ENC.decode(ENC.encode(over[3])[-50:]))
print("---")
print(ENC.decode(ENC.encode(over[4])[:50]))

no overlap : 185 chunks
50 overlap : 246 chunks  (+33% to store and embed)

The last 50 tokens of chunk 3 reappear at the start of chunk 4:
---
, she tried to look down and make out
what she was coming to, but it was too dark to see anything; then she
looked at the sides of the well, and noticed that they were filled with
cupboards and book-sh
---
, she tried to look down and make out
what she was coming to, but it was too dark to see anything; then she
looked at the sides of the well, and noticed that they were filled with
cupboards and book-sh


## 5. Embed and search

No vector database. With a few hundred chunks, searching is one dot product
against a small matrix — a database here would be pure overhead.

In [9]:
def embed(texts, batch=256):
    out = []
    for i in range(0, len(texts), batch):
        r = ai.embeddings.create(model="text-embedding-3-small",
                                 input=texts[i : i + batch])
        out.extend(d.embedding for d in r.data)
    V = np.asarray(out, dtype=np.float32)
    V /= np.linalg.norm(V, axis=1, keepdims=True)   # unit vectors -> cosine == dot
    return V


INDEX = {}


def build(name, chunks):
    """Embed a chunking once and remember it."""
    if name not in INDEX:
        print(f"embedding {name} ({len(chunks)} chunks, ~${sum(map(ntok, chunks))/1e6*0.02:.4f})")
        INDEX[name] = (chunks, embed(chunks))
    return INDEX[name]


def search(query, name, k=3):
    chunks, V = INDEX[name]
    q = embed([query])[0]
    sims = V @ q
    top = np.argsort(-sims)[:k]
    return [(float(sims[i]), chunks[i]) for i in top]

In [10]:
build("fixed-200", chunk_fixed(book, 200))
build("paragraphs-200", chunk_paragraphs(book, 200))
print("\nready:", list(INDEX))

embedding fixed-200 (185 chunks, ~$0.0007)
embedding paragraphs-200 (224 chunks, ~$0.0007)

ready: ['fixed-200', 'paragraphs-200']


## 6. Ask it something

You know the answer already — so just read what comes back and judge it.

In [11]:
def show(query, name, k=3, width=400):
    print("=" * 72)
    print(f"Q: {query}      [{name}]")
    print("=" * 72)
    for score, text in search(query, name, k):
        snippet = " ".join(text.split())[:width]
        print(f"\n[{score:.3f}] {snippet}...")


show("What happens at the Mad Hatter's tea party?", "paragraphs-200")

Q: What happens at the Mad Hatter's tea party?      [paragraphs-200]

[0.616] She had not gone much farther before she came in sight of the house of the March Hare: she thought it must be the right house, because the chimneys were shaped like ears and the roof was thatched with fur. It was so large a house, that she did not like to go nearer till she had nibbled some more of the lefthand bit of mushroom, and raised herself to about two feet high: even then she walked up tow...

[0.570] There was a table set out under a tree in front of the house, and the March Hare and the Hatter were having tea at it: a Dormouse was sitting between them, fast asleep, and the other two were using it as a cushion, resting their elbows on it, and talking over its head. “Very uncomfortable for the Dormouse,” thought Alice; “only, as it’s asleep, I suppose it doesn’t mind.” The table was a large one...

[0.568] “Well, I’d hardly finished the first verse,” said the Hatter, “when the Queen jumped up and bawl

In [12]:
show("Who is the Cheshire Cat and what does it do?", "paragraphs-200")

Q: Who is the Cheshire Cat and what does it do?      [paragraphs-200]

[0.602] She was looking about for some way of escape, and wondering whether she could get away without being seen, when she noticed a curious appearance in the air: it puzzled her very much at first, but, after watching it a minute or two, she made it out to be a grin, and she said to herself “It’s the Cheshire Cat: now I shall have somebody to talk to.” “How are you getting on?” said the Cat, as soon as ...

[0.544] So she set the little creature down, and felt quite relieved to see it trot away quietly into the wood. “If it had grown up,” she said to herself, “it would have made a dreadfully ugly child: but it makes rather a handsome pig, I think.” And she began thinking over other children she knew, who might do very well as pigs, and was just saying to herself, “if one only knew the right way to change the...

[0.540] “Cheshire Puss,” she began, rather timidly, as she did not at all know whether it would like th

In [13]:
show("What does the Queen of Hearts keep shouting?", "paragraphs-200")

Q: What does the Queen of Hearts keep shouting?      [paragraphs-200]

[0.580] “My name is Alice, so please your Majesty,” said Alice very politely; but she added, to herself, “Why, they’re only a pack of cards, after all. I needn’t be afraid of them!” “And who are _these?_” said the Queen, pointing to the three gardeners who were lying round the rose-tree; for, you see, as they were lying on their faces, and the pattern on their backs was the same as the rest of the pack, s...

[0.572] The King and Queen of Hearts were seated on their throne when they arrived, with a great crowd assembled about them—all sorts of little birds and beasts, as well as the whole pack of cards: the Knave was standing before them, in chains, with a soldier on each side to guard him; and near the King was the White Rabbit, with a trumpet in one hand, and a scroll of parchment in the other. In the very m...

[0.547] The Queen had only one way of settling all difficulties, great or small. “Off with his head!” s

## 7. The experiment: does chunk size matter?

Same question, same book, same embedding model — only the chunk size changes.
Read the results and decide for yourself which is most useful to hand a
language model as context.

Small chunks are precise but may cut off the surrounding context. Large
chunks carry plenty of context but bury the answer in unrelated text.

In [14]:
QUESTION = "What happens at the Mad Hatter's tea party?"

for size in [100, 200, 400, 800]:
    name = f"para-{size}"
    build(name, chunk_paragraphs(book, size))
    score, text = search(QUESTION, name, k=1)[0]
    snippet = " ".join(text.split())
    print("=" * 72)
    print(f"chunk size {size:>4} tok  |  {len(INDEX[name][0]):>3} chunks  |  "
          f"top score {score:.3f}  |  this chunk is {ntok(text)} tok")
    print("=" * 72)
    print(snippet[:600] + ("..." if len(snippet) > 600 else ""))
    print()

embedding para-100 (407 chunks, ~$0.0007)
chunk size  100 tok  |  407 chunks  |  top score 0.693  |  this chunk is 8 tok
CHAPTER VII. A Mad Tea-Party

embedding para-200 (224 chunks, ~$0.0007)
chunk size  200 tok  |  224 chunks  |  top score 0.616  |  this chunk is 143 tok
She had not gone much farther before she came in sight of the house of the March Hare: she thought it must be the right house, because the chimneys were shaped like ears and the roof was thatched with fur. It was so large a house, that she did not like to go nearer till she had nibbled some more of the lefthand bit of mushroom, and raised herself to about two feet high: even then she walked up towards it rather timidly, saying to herself “Suppose it should be raving mad after all! I almost wish I’d gone to see the Hatter instead!” CHAPTER VII. A Mad Tea-Party

embedding para-400 (104 chunks, ~$0.0007)
chunk size  400 tok  |  104 chunks  |  top score 0.682  |  this chunk is 381 tok
CHAPTER VII. A Mad Tea-Party There w

### What just happened — read this before moving on

Look at the **100-token** result. It has the *highest* similarity score of all
four (0.693), and it is completely useless: the chunk is eight tokens long and
contains nothing but the chapter heading, `CHAPTER VII. A Mad Tea-Party`.

Of course it scored well — a heading about a tea party is an excellent
semantic match for a question about the tea party. It just contains no
information. **A high similarity score does not mean a useful chunk.**

Now look at the other three:

| chunk size | what came back |
| --- | --- |
| 100 | the chapter title, and nothing else |
| 200 | Alice *approaching* the house — next to the scene, not in it |
| **400** | **the actual tea party** — table, March Hare, Hatter, Dormouse |
| 800 | starts with the Cheshire Cat; the tea party is buried further down |

400 tokens wins here, and you can see *why* rather than take it on faith. Too
small and the chunk matches without saying anything; too large and the answer
is diluted by whatever else got swept in.

This is also why the score alone is a bad guide, and why the
[science_qa.ipynb](science_qa.ipynb) harness scores by whether the *evidence
text* was actually delivered.

### Fixed-size vs paragraph boundaries

Same question, same target size — the only difference is whether cuts respect
sentence boundaries.

In [ ]:
build("fixed-200", chunk_fixed(book, 200))

for name in ["fixed-200", "paragraphs-200"]:
    score, text = search(QUESTION, name, k=1)[0]
    print("=" * 72)
    print(f"{name}   score {score:.3f}")
    print("=" * 72)
    print(" ".join(text.split())[:500])
    print()

## 8. Your turn

Change the question and re-run. Things worth trying:

- A question whose answer is spread over several pages
  (*"how does Alice change size throughout the story?"*) — does any single
  chunk contain it?
- A question about something **not** in the book (*"what is the capital of
  France?"*) — look at the similarity scores. Embedding search always returns
  *something*; the score is your only hint that nothing relevant exists.
- A very specific quote (*"off with her head"*) versus a vague theme
  (*"loneliness"*). Which does embedding search handle better?

In [ ]:
MY_QUESTION = "How does Alice change size throughout the story?"

show(MY_QUESTION, "paragraphs-200", k=3)

In [ ]:
# Nothing in the book answers this -- watch the scores, not the text.
show("What is the capital of France?", "paragraphs-200", k=3)

## Where to go next

- **Swap the book.** Same code, different ID: Sherlock Holmes `1661`,
  Frankenstein `84`, Pride and Prejudice `1342`.
- **Many documents instead of one** — Simple English Wikipedia has a flat
  `{id, url, title, text}` schema and adds the problem of picking the right
  *document* before picking the right chunk.
- **Measure instead of eyeball** — [rag_chunk_embed.md](rag_chunk_embed.md)
  and [science_qa.ipynb](science_qa.ipynb) do this properly, with a dataset
  that has labelled evidence. The key idea there: compare chunk sizes at a
  fixed **context budget**, not a fixed number of chunks, or bigger chunks
  win automatically.